# 26 · A8 test 1: distances by geometry in Methods 1 and 2 (no retraining)

Fixed in `PLAN.md` (amendment A8) before this notebook ran. Matt: *"distances should be handled with geometry
always, never wording. Test all 3"* and *"All methods should use geometry"*.

- **Zero-shot land map:** each pixel's projected vector (notebook 11's model) against the wording-0 phrase of every
  leaf class except "other"; the pixel takes the most similar phrase's class. Distance rules = `within` on it (30 m).
- **Method 1-G:** the plan in caption wording without its distance rules (land, patch size, climate as text);
  score = similarity + 2 × passes every distance rule.
- **Method 2-G:** the plan's land wording; score = similarity + 2 × (passes every distance rule and lies in a NASA
  POWER cell that meets every climate condition).
- **Information, Method 1-R:** Method 1-G's text with the distance wording kept, no geometry.
- **References:** Methods 1 (published: the whole question as text), 2 and 3 exactly as notebook 25; they must
  reproduce `25_scores.csv` within 1e-6 first.
- **Pass marks:** G1 (2-G vs 2) and G1' (1-G vs published 1): on the 20 distance questions a median F1 gain ≥ +0.10
  with 95% lower bound > 0 (paired bootstrap of questions, difference of medians), and on all 40 a median F1 no
  lower than the reference's − 0.01.

The next four cells are notebook 12's loading code, unchanged.

In [1]:
import json
import time

import numpy as np
import pandas as pd

from askmap.catalogue import area_classes
from askmap.config import PROJECT_ROOT, data_root, load_config
from askmap.layers import SOLAR_LATER, official_leaf_map
from askmap.plan import AreaContext, execute, validate
from askmap.power import add_derived, fetch_daily, pixel_cells
from askmap.tiles import area_tiles, pixel_lonlat
from askmap.universe import Universe, box_of

cfg = load_config(PROJECT_ROOT / "configs" / "feasibility.yaml")
Q = load_config(PROJECT_ROOT / "configs" / "queries.yaml")["queries"]
RW = load_config(PROJECT_ROOT / "configs" / "reworded.yaml")["reworded"]
ROOT = data_root()
RES = PROJECT_ROOT / "outputs" / "results"
FIG = PROJECT_ROOT / "outputs" / "figures"
CL, P, SC = cfg["classes"], cfg["power"], cfg["scoring"]
LEAVES, GROUPS = CL["leaves"], CL["groups"]
CATALOGUE = area_classes(cfg)
T_START = time.time()
cells = pd.read_parquet(ROOT / "power" / "universe_cells.parquet")
daily = {
    box: add_derived(fetch_daily(P, b, ROOT / "power" / f"universe_{box}_daily.nc"), P["low_wind_sun_pct"])
    for box, b in P["boxes"].items()
}
uni = Universe(cells=cells, daily=daily)
SEEN = ["north_dakota", "north_carolina"]
HELD = cfg["heldout_pools"]  # seen area -> held-out state
POOL_Q = {
    "north_dakota": "north_dakota",
    "north_carolina": "north_carolina",
    "minnesota": "north_dakota",
    "georgia": "north_carolina",
}  # pool -> area whose questions it answers
T = {}
for area in POOL_Q:
    for name, (tile, role) in area_tiles(cfg["areas"][area], cfg["grid"]).items():
        z = np.load(ROOT / "tiles" / f"{area}_{name}.npz")
        solar = z["solar"] if "solar" in z.files else None
        nwi = z["nwi"] if "nwi" in z.files else None
        valid = np.isfinite(z["ae"]).all(axis=0)
        if solar is not None:
            valid &= solar != SOLAR_LATER
        lon, lat = pixel_lonlat(tile)
        box = box_of(float(lon.mean()), float(lat.mean()), P["boxes"])
        iy, ix = pixel_cells(daily[box].lat.values, daily[box].lon.values, lat, lon)
        T[name] = {
            "area": area,
            "role": role,
            "valid": valid,
            "box": box,
            "iy": iy,
            "ix": ix,
            "cell": uni.index(box, iy, ix),
            "official": official_leaf_map(z["cdl"], LEAVES, CL, nwi=nwi, solar=solar),
        }
        if role == "test":
            T[name]["ae_map"] = np.load(ROOT / "maps" / f"ae_{name}.npz")["leaf"]
            T[name]["planted_map"] = np.load(ROOT / "maps" / f"planted_{name}.npz")["leaf"]
CTX, TILES = {}, {}
for area in POOL_Q:
    names = [n for n, t in T.items() if t["area"] == area]
    ds = daily[T[names[0]]["box"]]
    ac = np.zeros((ds.sizes["lat"], ds.sizes["lon"]), bool)
    for n in names:
        ac[T[n]["iy"], T[n]["ix"]] = True
    CTX[area] = AreaContext(
        leaves=LEAVES,
        groups=GROUPS,
        res_m=cfg["grid"]["res_m"],
        power=ds,
        area_cells=ac,
        pixel_cell={n: (T[n]["iy"], T[n]["ix"]) for n in names},
    )
    TILES[area] = {
        "train": [n for n in names if T[n]["role"] == "train"],
        "test": [n for n in names if T[n]["role"] == "test"],
    }


def answer(plan, pool, maps="official"):
    """Answer of a plan in a pool's test tiles, as one vector over their valid pixels (fixed tile order)."""
    tiles = TILES[pool]["test"]
    key = "official" if maps == "official" else f"{maps}_map"
    got = execute(plan, {n: T[n][key] for n in tiles}, CTX[pool])
    return np.concatenate([got[n][T[n]["valid"]] for n in tiles])


def question_text(q, wording, pool):
    text = q["question"] if wording != "reworded" else RW[q["id"]]
    if pool in ("minnesota", "georgia"):
        text = text.replace("North Dakota", "Minnesota").replace("North Carolina", "Georgia")
    return text


def n_extra(plan):
    return len(plan["near"]) + len(plan["climate"]) + int(plan["land"]["min_patch_ha"] is not None)


print(f"{len(T)} tiles loaded; pools {dict((a, TILES[a]['test']) for a in POOL_Q)}")

18 tiles loaded; pools {'north_dakota': ['Kidder', 'Foster', 'Eddy'], 'north_carolina': ['NC4', 'NC5', 'NC6'], 'minnesota': ['MN1', 'MN2', 'MN3'], 'georgia': ['GA1', 'GA2', 'GA3']}


In [2]:
import os

import torch
from sklearn.metrics import average_precision_score

from askmap.aether_us import Projector, climate_ok_tile, plan_scores_tile, similarities, text_vectors
from askmap.captions import render_plan

AU = load_config(PROJECT_ROOT / "configs" / "aether_us.yaml")
CAP = load_config(PROJECT_ROOT / "configs" / "captions.yaml")
MCFG = AU["model"]
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
V11 = json.loads((RES / "11_verdicts.json").read_text())
BEST = V11["best_encoder"]
KIND = {"clip": "clip", "modern": "sentence_transformer"}[BEST]
ENC_NAME = AU["text_encoders"][BEST]
OUT = ROOT / "aether_us"
ck = torch.load(OUT / f"projector_{BEST}.pt", weights_only=False)
MU, SD = ck["mu"], ck["sd"]
TXT = np.load(OUT / f"text_{BEST}.npz")
TPOS = {s: i for i, s in enumerate(TXT["strings"])}
TVEC = TXT["vectors"].astype(np.float32)
model = Projector(len(MU), TVEC.shape[1], MCFG["layers"], MCFG["hidden"]).to(device)
model.load_state_dict(ck["state"])
SUMMARY = np.load(ROOT / "climate" / "enc_summary.npy")
HELDOUT = ["minnesota", "georgia"]
NB = 2000


def unit(v):
    return v / np.linalg.norm(v, axis=-1, keepdims=True)


def med_ci(v, seed=0):
    v = np.asarray(v, float)
    r = np.random.default_rng(seed)
    b = [np.median(v[r.integers(0, len(v), len(v))]) for _ in range(NB)]
    return float(np.median(v)), float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))


def topk_f1(truth, score):
    k = int(truth.sum())
    return float(truth[np.argpartition(-score, k)[:k]].mean())


print(f"best encoder from notebook 11: {BEST} ({ENC_NAME})")

best encoder from notebook 11: clip (openai/clip-vit-large-patch14)


In [3]:
HP = {}
for pool in HELDOUT:
    ae, cl = [], []
    for n in TILES[pool]["test"]:
        flat = np.flatnonzero(T[n]["valid"].ravel())
        z = np.load(ROOT / "tiles" / f"{T[n]['area']}_{n}.npz")["ae"]
        ae.append(z.reshape(64, -1)[:, flat].T.astype(np.float32))
        cl.append(T[n]["cell"].ravel()[flat])
    HP[pool] = {"ae": np.concatenate(ae), "cell": np.concatenate(cl)}
TRUTH = dict(np.load(ROOT / "maps" / "truth.npz"))
LLM_DIR = PROJECT_ROOT / "outputs" / "llm" / "main"
rows = []
for q in Q:
    pool = HELD[q["plan"]["area"]]
    n = len(HP[pool]["cell"])
    g = np.unpackbits(TRUTH[f"{q['id']}|{pool}"])[:n].astype(bool)
    if g.sum() < SC["min_truth"]:
        continue
    rec = json.loads((LLM_DIR / f"as_written_{pool}_{q['id']}.json").read_text())
    plan, err = rec["plan"], rec["error"]
    if plan is not None:
        try:
            validate(plan, CATALOGUE, (2001, 2025))
            if plan["area"] != pool:
                plan, err = None, "wrong area"
        except (ValueError, KeyError, TypeError) as e:
            plan, err = None, f"invalid plan: {e}"
    rows.append(
        {
            "id": q["id"],
            "pool": pool,
            "text": question_text(q, "as_written", pool),
            "truth": g,
            "has_climate": bool(q["plan"]["climate"]),
            "plan": plan,
            "plan_error": err,
        }
    )
QS = rows
print(
    f"{len(QS)} scored held-out questions; {sum(r['plan'] is None for r in QS)} without a usable plan; "
    f"{sum(r['has_climate'] for r in QS)} with a climate condition"
)

40 scored held-out questions; 1 without a usable plan; 23 with a climate condition


In [4]:
for r in QS:
    r["land_text"] = render_plan(dict(r["plan"], climate=[]), CAP, None) if r["plan"] is not None else None
new = sorted({r["land_text"] for r in QS if r["land_text"] is not None and r["land_text"] not in TPOS})
NEWVEC = (
    dict(zip(new, text_vectors(new, ENC_NAME, KIND, device, os.environ["HF_HOME"]), strict=True))
    if new
    else {}
)


def tvec(s):
    return TVEC[TPOS[s]] if s in TPOS else NEWVEC[s]


pd.DataFrame(
    [
        {"id": r["id"], "pool": r["pool"], "question": r["text"], "land_text": r["land_text"]}
        for r in QS
        if r["has_climate"]
    ]
).head(12)

/Volumes/disco2/github/climate-embeddings-crop-integrity/ask-the-map/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1029 13:53:58.499000 5481 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Loading weights:  78%|███████▊  | 458/590 [00:00<00:00, 4575.49it/s]

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 4713.19it/s]

,id,pool,question,land_text
0,nd09,minnesota,How much cropland in the Minnesota study area ...,Show cropland
1,nd15,minnesota,"In the Minnesota study area, show grassland an...",Show grassland and pasture
2,nd16,minnesota,"In the Minnesota study area, show grassland an...",Show grassland and pasture
3,nd18,minnesota,"In the Minnesota study area, show soybean fiel...",Show soybean fields
4,nd19,minnesota,"In the Minnesota study area, show spring wheat...",Show spring wheat
5,nd20,minnesota,How much cropland in the Minnesota study area ...,Show cropland
6,nd21,minnesota,"In the Minnesota study area, show sunflower fi...",NaN
7,nd23,minnesota,"In the Minnesota study area, show cropland in ...",Show cropland
8,nd24,minnesota,"In the Minnesota study area, show corn fields ...",Show corn fields
9,nd25,minnesota,"In the Minnesota study area, show cropland whe...",Show cropland


## Method 3's classifiers (notebook 12, loaded) for the reproduction check

In [5]:
from askmap.score import paired_median_gap
from askmap.zeroshot import distance_ok, leaf_grid, leaf_phrases, plan_text, zero_shot_leaf

TC = np.load(OUT / "tag_classifiers.npz")
if list(TC["leaves"]) != LEAVES:
    raise ValueError("tag_classifiers.npz leaves differ from the configured leaves")
COEF, ICPT = TC["coef"], TC["intercept"]


def probs_tile(n):
    ae = np.load(ROOT / "tiles" / f"{T[n]['area']}_{n}.npz")["ae"].astype(np.float32)
    lin = np.einsum("lc,chw->lhw", COEF, np.nan_to_num(ae)) + ICPT[:, None, None]
    return 1 / (1 + np.exp(-lin))


PROBS = {n: probs_tile(n) for pool in HELDOUT for n in TILES[pool]["test"]}
CAND, PHRASES = leaf_phrases(CAP, LEAVES)
print("zero-shot candidates:", dict(zip([LEAVES[i] for i in CAND], PHRASES, strict=True)))

zero-shot candidates: {'corn': 'corn fields', 'soybeans': 'soybean fields', 'spring_wheat': 'spring wheat', 'sunflower': 'sunflower fields', 'canola': 'canola fields', 'cotton': 'cotton fields', 'other_crop': 'other annual crops', 'hay': 'hay land', 'grassland': 'grassland and pasture', 'fallow': 'fallow or idle cropland', 'forest': 'forest', 'developed': 'developed land', 'water': 'open water', 'wetland': 'wetlands', 'solar_farm': 'existing solar farms'}


## Texts for every method, and the aligned space's similarities (questions, texts and the 15 land phrases)

In [6]:
for r in QS:
    p = r["plan"]
    r["text_1g"] = plan_text(p, CAP, distances=False, climate=True) if p is not None else None
    r["text_2g"] = plan_text(p, CAP, distances=False, climate=False) if p is not None else None
    r["text_1r"] = plan_text(p, CAP, distances=True, climate=True) if p is not None else None
extra = sorted(
    {r[k] for r in QS for k in ("text_1g", "text_2g", "text_1r") if r[k] is not None} | set(PHRASES)
)
extra = [s for s in extra if s not in TPOS and s not in NEWVEC]
NEWVEC |= dict(zip(extra, text_vectors(extra, ENC_NAME, KIND, device, os.environ["HF_HOME"]), strict=True))
ZS = {}  # tile -> zero-shot leaf map (30 m)
for pool in HELDOUT:
    mine = [r for r in QS if r["pool"] == pool]
    texts = list(PHRASES) + sorted(
        {
            r[k]
            for r in mine
            for k in ("text", "land_text", "text_1g", "text_2g", "text_1r")
            if r[k] is not None
        }
    )
    sims = similarities(
        model,
        HP[pool]["ae"],
        HP[pool]["cell"],
        SUMMARY,
        MU,
        SD,
        unit(np.array([tvec(t) for t in texts])),
        device,
    )
    row_of = {t: i for i, t in enumerate(texts)}
    leaf = zero_shot_leaf(sims[: len(PHRASES)], CAND)
    start = 0
    for n in TILES[pool]["test"]:
        k = int(T[n]["valid"].sum())
        ZS[n] = leaf_grid(T[n]["valid"], leaf[start : start + k])
        start += k
    for r in mine:
        for k in ("text", "land_text", "text_1g", "text_2g", "text_1r"):
            r[f"sim_{k}"] = sims[row_of[r[k]]] if r[k] is not None else None
    del sims
print({n: int((m >= 0).sum()) for n, m in ZS.items()})

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 8699.49it/s]

{'MN1': 444869, 'MN2': 444889, 'MN3': 444889, 'GA1': 444889, 'GA2': 444889, 'GA3': 444889}


## Zero-shot land map against the official map (held-out test tiles, 30 m; information)

In [7]:
rows = []
for li in CAND:
    pred = np.concatenate([(ZS[n] == li)[T[n]["valid"]] for pool in HELDOUT for n in TILES[pool]["test"]])
    true = np.concatenate(
        [(T[n]["official"] == li)[T[n]["valid"]] for pool in HELDOUT for n in TILES[pool]["test"]]
    )
    tp = int((pred & true).sum())
    rows.append(
        {
            "class": LEAVES[li],
            "official_px": int(true.sum()),
            "mapped_px": int(pred.sum()),
            "precision": tp / pred.sum() if pred.any() else np.nan,
            "recall": tp / true.sum() if true.any() else np.nan,
        }
    )
ZS_ACC = pd.DataFrame(rows)
ZS_ACC["f1"] = 2 * ZS_ACC.precision * ZS_ACC.recall / (ZS_ACC.precision + ZS_ACC.recall)
ZS_ACC.to_csv(RES / "26_zero_shot_map.csv", index=False)
ZS_ACC.round(3)

,class,official_px,mapped_px,precision,recall,f1
0,corn,284594,269027,0.942,0.890,0.916
1,soybeans,161328,143502,0.927,0.825,0.873
2,spring_wheat,505,365,0.134,0.097,0.113
3,sunflower,134,2963,0.000,0.000,NaN
4,canola,6,0,NaN,0.000,NaN
5,cotton,64721,78987,0.742,0.906,0.816
6,other_crop,63768,62858,0.711,0.701,0.706
7,hay,30658,95559,0.300,0.936,0.455
8,grassland,463959,351866,0.700,0.531,0.604
9,fallow,2667,23600,0.065,0.579,0.118


## Score every question

In [8]:
REF = {q["id"]: q["plan"] for q in Q}
out = []
for r in QS:
    pool, g = r["pool"], r["truth"]
    row = {
        "id": r["id"],
        "pool": pool,
        "question": r["text"],
        "has_climate": r["has_climate"],
        "has_distance": bool(REF[r["id"]]["near"]),
        "plan_ok": r["plan"] is not None,
        "truth_px": int(g.sum()),
        "prevalence": float(g.mean()),
        "F1_m1": topk_f1(g, r["sim_text"]),
        "AP_m1": average_precision_score(g, r["sim_text"]),
    }
    if r["plan"] is None:
        for k in ("m2", "m3", "m1g", "m2g", "m1r"):
            row[f"F1_{k}"], row[f"AP_{k}"] = 0.0, row["prevalence"]
    else:
        tiles, plan = TILES[pool]["test"], r["plan"]
        clim = np.concatenate([climate_ok_tile(plan, n, CTX[pool])[T[n]["valid"]] for n in tiles])
        dist = np.concatenate([distance_ok(plan, ZS[n], CTX[pool])[T[n]["valid"]] for n in tiles])
        scores = {
            "m2": r["sim_land_text"] + 2.0 * clim,
            "m3": np.concatenate(
                [plan_scores_tile(plan, PROBS[n], n, CTX[pool])[T[n]["valid"]] for n in tiles]
            ),
            "m1g": r["sim_text_1g"] + 2.0 * dist,
            "m2g": r["sim_text_2g"] + 2.0 * (dist & clim),
            "m1r": r["sim_text_1r"],
        }
        for k, s in scores.items():
            row[f"F1_{k}"], row[f"AP_{k}"] = topk_f1(g, s), average_precision_score(g, s)
        row["distance_pass_share"] = float(dist.mean())
    row |= {"text_1g": r["text_1g"], "text_2g": r["text_2g"]}
    out.append(row)
S26 = pd.DataFrame(out)
S26.to_csv(RES / "26_scores.csv", index=False)
print(len(S26), "questions;", int(S26.has_climate.sum()), "climate;", int(S26.has_distance.sum()), "distance")

40 questions; 23 climate; 20 distance


## Reproduction check against notebook 25 (must hold before anything is reported)

In [9]:
S25 = pd.read_csv(RES / "25_scores.csv")
m = S26.merge(S25, on=["id", "pool"], suffixes=("", "_25"), validate="one_to_one")
if len(m) != len(S25) or len(m) != len(S26):
    raise AssertionError("the question sets of notebooks 25 and 26 differ")
for k in ("F1_m1", "AP_m1", "F1_m2", "AP_m2", "F1_m3", "AP_m3"):
    worst = float((m[k] - m[f"{k}_25"]).abs().max())
    print(f"{k}: largest difference {worst:.2e}")
    if worst > 1e-6:
        raise AssertionError(f"{k} does not reproduce notebook 25")
print("Methods 1, 2 and 3 reproduce notebook 25")

F1_m1: largest difference 0.00e+00
AP_m1: largest difference 0.00e+00
F1_m2: largest difference 0.00e+00
AP_m2: largest difference 0.00e+00
F1_m3: largest difference 0.00e+00
AP_m3: largest difference 0.00e+00
Methods 1, 2 and 3 reproduce notebook 25


## Results and pass marks

In [10]:
NBOOT = 2000
dist_q, clim_q = S26[S26.has_distance], S26[S26.has_climate]


def gap(df, a, b, kind="F1"):
    g, lo, hi = paired_median_gap(df[f"{kind}_{a}"].to_numpy(), df[f"{kind}_{b}"].to_numpy(), NBOOT, 0)
    return {"gap": g, "lo": lo, "hi": hi}


KEYS = ("m1", "m1g", "m1r", "m2", "m2g", "m3")
R = {
    "n": len(S26),
    "distance_n": len(dist_q),
    "climate_n": len(clim_q),
    "median_F1": {k: float(S26[f"F1_{k}"].median()) for k in KEYS},
    "distance_median_F1": {k: float(dist_q[f"F1_{k}"].median()) for k in KEYS},
    "climate_median_AP": {k: float(clim_q[f"AP_{k}"].median()) for k in KEYS},
    "gaps": {
        "m2g_minus_m2_distance": gap(dist_q, "m2g", "m2"),
        "m2g_minus_m2_all": gap(S26, "m2g", "m2"),
        "m1g_minus_m1_distance": gap(dist_q, "m1g", "m1"),
        "m1g_minus_m1_all": gap(S26, "m1g", "m1"),
        "m1g_minus_m1r_distance": gap(dist_q, "m1g", "m1r"),
        "m1r_minus_m1_all": gap(S26, "m1r", "m1"),
        "m3_minus_m2g_all": gap(S26, "m3", "m2g"),
        "m3_minus_m2g_distance": gap(dist_q, "m3", "m2g"),
        "m2g_minus_m1g_all": gap(S26, "m2g", "m1g"),
        "m2g_minus_m2_climate_AP": gap(clim_q, "m2g", "m2", "AP"),
    },
}


def mark(new, ref):
    d, a = R["gaps"][f"{new}_minus_{ref}_distance"], R["median_F1"]
    return {
        "distance_gain_ok": d["gap"] >= 0.10 and d["lo"] > 0,
        "all_ok": a[new] >= a[ref] - 0.01,
        "pass": d["gap"] >= 0.10 and d["lo"] > 0 and a[new] >= a[ref] - 0.01,
    }


R["G1_m2g_vs_m2"] = mark("m2g", "m2")
R["G1prime_m1g_vs_m1"] = mark("m1g", "m1")
(RES / "26_verdicts.json").write_text(json.dumps(R, indent=1))
print(json.dumps(R, indent=1))

{
 "n": 40,
 "distance_n": 20,
 "climate_n": 23,
 "median_F1": {
  "m1": 0.26015978990881916,
  "m1g": 0.5222917662725621,
  "m1r": 0.2935264124748659,
  "m2": 0.4956257294287543,
  "m2g": 0.6126980361989016,
  "m3": 0.5619548616534145
 },
 "distance_median_F1": {
  "m1": 0.0910727551724547,
  "m1g": 0.39067343613465,
  "m1r": 0.11141767728626964,
  "m2": 0.11141767728626964,
  "m2g": 0.4473649508019705,
  "m3": 0.3449688408491657
 },
 "climate_median_AP": {
  "m1": 0.28968077499151984,
  "m1g": 0.5787878629056987,
  "m1r": 0.5604533239072852,
  "m2": 0.7374035610549866,
  "m2g": 0.7430140913864701,
  "m3": 0.7730540284217732
 },
 "gaps": {
  "m2g_minus_m2_distance": {
   "gap": 0.33594727351570086,
   "lo": 0.13512682937254455,
   "hi": 0.43929831159784083
  },
  "m2g_minus_m2_all": {
   "gap": 0.1170723067701473,
   "lo": 0.0,
   "hi": 0.4137075334045829
  },
  "m1g_minus_m1_distance": {
   "gap": 0.2996006809621953,
   "lo": 0.16112592485392283,
   "hi": 0.4370699203757587
  },
  "m

In [11]:
pd.set_option("display.max_colwidth", 70)
S26[
    [
        "id",
        "pool",
        "question",
        "has_distance",
        "has_climate",
        "F1_m1",
        "F1_m1g",
        "F1_m1r",
        "F1_m2",
        "F1_m2g",
        "F1_m3",
    ]
].round(3)

,id,pool,question,has_distance,has_climate,F1_m1,F1_m1g,F1_m1r,F1_m2,F1_m2g,F1_m3
0,nd01,minnesota,"In the Minnesota study area, show cropland within 100 m of a wetland.",True,False,0.069,0.353,0.110,0.110,0.353,0.321
1,nd02,minnesota,"In the Minnesota study area, show cropland within 30 m of a wetland.",True,False,0.005,0.213,0.022,0.022,0.213,0.115
2,nd03,minnesota,How much cropland in the Minnesota study area is within 100 m of a...,True,False,0.098,0.353,0.110,0.110,0.353,0.321
3,nd04,minnesota,"In the Minnesota study area, show cropland that is more than 300 m...",True,False,0.598,0.822,0.733,0.733,0.822,0.742
4,nd05,minnesota,"In the Minnesota study area, show wetlands within 100 m of cropland.",True,False,0.141,0.467,0.143,0.143,0.467,0.394
5,nd09,minnesota,How much cropland in the Minnesota study area got at least 250 mm ...,False,True,0.913,0.928,0.928,0.927,0.927,0.928
6,nd15,minnesota,"In the Minnesota study area, show grassland and pasture where June...",False,True,0.575,0.594,0.594,0.673,0.673,0.649
7,nd16,minnesota,"In the Minnesota study area, show grassland and pasture in the dri...",False,True,0.244,0.236,0.236,0.665,0.665,0.620
8,nd18,minnesota,"In the Minnesota study area, show soybean fields where August 2023...",False,True,0.646,0.667,0.667,0.919,0.919,0.917
9,nd19,minnesota,"In the Minnesota study area, show spring wheat where June-July 202...",False,True,0.126,0.139,0.139,0.174,0.174,0.021


In [12]:
print(f"Run time {(time.time() - T_START) / 60:.1f} min")

Run time 2.3 min
